In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_case_code_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "CaseCodeId	string ,"
        #     "SourceSystem	string,"
        #     "PortfolioKey	string,"
        #     "PortfolioName	string,"
        #     "IngestionTimestamp	timestamp,"
        #     "BatchId	string,"
        #     "CaseCodeKey	int,"
        #     "CaseCodePrimary	int,"
        #     "CaseRelationTypeCode	int,"
        #     "CustCode	int,"
        #     "DebtCode	int,"
        #     "QueueCode	int,"
        #     "Queue	string,"
        #     "AdminStatusCode	int,"
        #     "AdminStatus	string,"
        #     "AdminStatusDate	date,"
        #     "AdminStatusExpirationDate	date,"
        #     "StrategyStepCode	int,"
        #     "StrategyStep	string,"
        #     "CaseSourceCode	string,"
        #     "CurrentlyOpenFlag	int,"
        #     "ParentStrategyStepCode	int,"
        #     "ParentStrategyStep	string,"
        #     "StrategyTypeCode	string,"
        #     "StrategyType	string,"
        #     "StrategyFlowCode	string,"
        #     "StrategyFlow	string,"
        #     "WatchlistFlag	string,"
        #     "SubsidyEndDate	timestamp,"
        #     "LegalFileLocation	string,"
        #     "ApplicationDecisionEverDate	timestamp,"
        #     "GovernmentSubsidyType	string,"
        #     "AgencyCode	int,"
        #     "Agency	string,"
        #     "PacketCode	int,"
        #     "Packet	string,"
        #     "AppealFlag	int,"
        #     "ConfiscationCurrentAgencyCode	int,"
        #     "ConfiscationCurrentAgency	string,"
        #     "CurrentStageConfiscationCode	int,"
        #     "CurrentStageConfiscation	string,"
        #     "ForcedPrenotationCurrentAgencyCode	int,"
        #     "ForcedPrenotationCurrentAgency	string,"
        #     "CurrentStageForcedPrenotationCode	int,"
        #     "CurrentStageForcedPrenotation	string,"
        #     "RegularSuitCurrentAgencyCode	int,"
        #     "RegularSuitCurrentAgency	string,"
        #     "CurrentStageRegularSuitCode	int,"
        #     "CurrentStageRegularSuit	string,"
        #     "PaymentOrderCurrentAgencyCode	int,"
        #     "PaymentOrderCurrentAgency	string,"
        #     "CurrentStagePaymentOrderCode	int,"
        #     "CurrentStagePaymentOrder	string,"
        #     "TransactionPerimeterCode	string,"
        #     "AccountNumber string,"
        #     "MinimumPaymentAmount decimal(19,4)"
        #     "AppealCode int,"
        #     "ActiveAppealFlag boolean,"
        #     "DecisionDate date,"
        #     "CommitteeDecisionTypeCode string,"
        #     "CommitteeDecisionType string,"
        #     "CaseCodeKeyOld int", #new
        #     "BusinessDate	date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.CaseCode"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "qca",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== fs_cases_snap ==================

        fs_cases_snap_extra_filters = None

        if fs_cases_snap_extra_filters is not None:
            fs_cases_snap_extra_filters.append(date_filter)
            fs_cases_snap_final_filters = " AND ".join(fs_cases_snap_extra_filters)
        else:
            fs_cases_snap_final_filters = date_filter

        fs_cases_snap_selected_fields = [
            "case_code",
            "case_relation_type_code",
            "cust_code",
            "debt_code",
            "que_code",
            "wfst_code_strategy_step",
            "case_source",
            "case_legal_process_status",
            "case_legal_process_status_activity_date",
            "caex_custom_datetime_28",
            "extc_code",
            "pack_code",
            "cass_generic_int_03",
            "SPET_CODE",
            "businessdate",
        ]

        fs_cases_snap = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap",
            fs_cases_snap_final_filters,
            fs_cases_snap_selected_fields,
        )

        # ================== dim_queue ==================

        dim_queue_extra_filters = None

        if dim_queue_extra_filters is not None:
            dim_queue_extra_filters.append(date_filter)
            dim_queue_final_filters = " AND ".join(dim_queue_extra_filters)
        else:
            dim_queue_final_filters = date_filter

        dim_queue_selected_fields = [
            "que_desc",
            "que_id",
        ]

        dim_queue = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_queue",
            dim_queue_final_filters,
            dim_queue_selected_fields,
        )

        # ================== dim_strategy_step ==================

        dim_strategy_step_extra_filters = None

        if dim_strategy_step_extra_filters is not None:
            dim_strategy_step_extra_filters.append(date_filter)
            dim_strategy_step_final_filters = " AND ".join(
                dim_strategy_step_extra_filters
            )
        else:
            dim_strategy_step_final_filters = date_filter

        dim_strategy_step_selected_fields = [
            "str_step_desc",
            "str_steph1_l3_id",
            "str_steph1_l3_desc",
            "str_steph3_desc",
            "str_steph1_desc",
            "str_step_id",
        ]

        dim_strategy_step = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_strategy_step",
            dim_strategy_step_final_filters,
            dim_strategy_step_selected_fields,
        )

        # ================== dim_cases_static ==================

        dim_cases_static_extra_filters = None

        if dim_cases_static_extra_filters is not None:
            dim_cases_static_extra_filters.append(date_filter)
            dim_cases_static_final_filters = " AND ".join(
                dim_cases_static_extra_filters
            )
        else:
            dim_cases_static_final_filters = date_filter

        dim_cases_static_selected_fields = [
            "case_currently_open",
            "cstm_datetime_ad_03",
            "case_account_for_search",
            "case_code",
            "case_code_primary",
            "CASE_CONTR_NUM",
            "CSTM_TEXT_AD_20", #new
        ]

        dim_cases_static = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_cases_static",
            dim_cases_static_final_filters,
            dim_cases_static_selected_fields,
        )

        # ================== dim_legal_process_status ==================

        dim_legal_process_status_extra_filters = None

        if dim_legal_process_status_extra_filters is not None:
            dim_legal_process_status_extra_filters.append(date_filter)
            dim_legal_process_status_final_filters = " AND ".join(
                dim_legal_process_status_extra_filters
            )
        else:
            dim_legal_process_status_final_filters = date_filter

        dim_legal_process_status_selected_fields = [
            "lepr_desc",
            "lepr_id",
        ]

        dim_legal_process_status = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_legal_process_status",
            dim_legal_process_status_final_filters,
            dim_legal_process_status_selected_fields,
        )

        # ================== fsad_cases_extra_open ==================

        fsad_cases_extra_open_extra_filters = None

        if fsad_cases_extra_open_extra_filters is not None:
            fsad_cases_extra_open_extra_filters.append(date_filter)
            fsad_cases_extra_open_final_filters = " AND ".join(
                fsad_cases_extra_open_extra_filters
            )
        else:
            fsad_cases_extra_open_final_filters = date_filter

        fsad_cases_extra_open_selected_fields = [
            "caex_custom_text_29",
            "case_code",
        ]

        fsad_cases_extra_open = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_cases_extra_open",
            fsad_cases_extra_open_final_filters,
            fsad_cases_extra_open_selected_fields,
        )

        # ================== fs_cases_snap_extra ==================

        fs_cases_snap_extra_extra_filters = None

        if fs_cases_snap_extra_extra_filters is not None:
            fs_cases_snap_extra_extra_filters.append(date_filter)
            fs_cases_snap_extra_final_filters = " AND ".join(
                fs_cases_snap_extra_extra_filters
            )
        else:
            fs_cases_snap_extra_final_filters = date_filter

        fs_cases_snap_extra_selected_fields = [
            "cext_generic_nvarchar_06",
            "cext_generic_nvarchar_24",
            "cext_generic_int_15",
            "cext_generic_int_16",
            "cext_generic_int_11",
            "cext_generic_int_12",
            "cext_generic_int_18",
            "cext_generic_int_19",
            "cext_generic_int_09",
            "cext_generic_int_10",
            "CEXT_DEDICATED_PROPERTY_01_DECIMAL",
            "case_code",
        ]

        fs_cases_snap_extra = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_fs_cases_snap_extra",
            fs_cases_snap_extra_final_filters,
            fs_cases_snap_extra_selected_fields,
        )

        # ================== dim_external_company ==================

        dim_external_company_extra_filters = None

        if dim_external_company_extra_filters is not None:
            dim_external_company_extra_filters.append(date_filter)
            dim_external_company_final_filters = " AND ".join(
                dim_external_company_extra_filters
            )
        else:
            dim_external_company_final_filters = date_filter

        dim_external_company_selected_fields = [
            "extc_name",
            "extc_id",
        ]

        dim_external_company = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_final_filters,
            dim_external_company_selected_fields,
        )

        # ================== dim_external_company_2 ==================

        dim_external_company_2_extra_filters = None

        if dim_external_company_2_extra_filters is not None:
            dim_external_company_2_extra_filters.append(date_filter)
            dim_external_company_2_final_filters = " AND ".join(
                dim_external_company_2_extra_filters
            )
        else:
            dim_external_company_2_final_filters = date_filter

        dim_external_company_2_selected_fields = [
            "extc_name",
            "extc_id",
        ]

        dim_external_company_2 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_2_final_filters,
            dim_external_company_2_selected_fields,
        )

        # ================== dim_external_company_3 ==================

        dim_external_company_3_extra_filters = None

        if dim_external_company_3_extra_filters is not None:
            dim_external_company_3_extra_filters.append(date_filter)
            dim_external_company_3_final_filters = " AND ".join(
                dim_external_company_3_extra_filters
            )
        else:
            dim_external_company_3_final_filters = date_filter

        dim_external_company_3_selected_fields = [
            "extc_name",
            "extc_id",
        ]

        dim_external_company_3 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_3_final_filters,
            dim_external_company_3_selected_fields,
        )

        # ================== dim_external_company_4 ==================

        dim_external_company_4_extra_filters = None

        if dim_external_company_4_extra_filters is not None:
            dim_external_company_4_extra_filters.append(date_filter)
            dim_external_company_4_final_filters = " AND ".join(
                dim_external_company_4_extra_filters
            )
        else:
            dim_external_company_4_final_filters = date_filter

        dim_external_company_4_selected_fields = [
            "extc_name",
            "extc_id",
        ]

        dim_external_company_4 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_4_final_filters,
            dim_external_company_4_selected_fields,
        )

        # ================== dim_external_company_5 ==================

        dim_external_company_5_extra_filters = None

        if dim_external_company_5_extra_filters is not None:
            dim_external_company_5_extra_filters.append(date_filter)
            dim_external_company_5_final_filters = " AND ".join(
                dim_external_company_5_extra_filters
            )
        else:
            dim_external_company_5_final_filters = date_filter

        dim_external_company_5_selected_fields = [
            "extc_name",
            "extc_id",
        ]

        dim_external_company_5 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_external_company",
            dim_external_company_5_final_filters,
            dim_external_company_5_selected_fields,
        )

        # ================== dim_packet ==================

        dim_packet_extra_filters = None

        if dim_packet_extra_filters is not None:
            dim_packet_extra_filters.append(date_filter)
            dim_packet_final_filters = " AND ".join(dim_packet_extra_filters)
        else:
            dim_packet_final_filters = date_filter

        dim_packet_selected_fields = [
            "pack_name",
            "pack_id",
            "packh3_id",
        ]

        dim_packet = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet",
            dim_packet_final_filters,
            dim_packet_selected_fields,
        )

        # ================== dim_process_stage_definition ==================

        dim_process_stage_definition_extra_filters = None

        if dim_process_stage_definition_extra_filters is not None:
            dim_process_stage_definition_extra_filters.append(date_filter)
            dim_process_stage_definition_final_filters = " AND ".join(
                dim_process_stage_definition_extra_filters
            )
        else:
            dim_process_stage_definition_final_filters = date_filter

        dim_process_stage_definition_selected_fields = [
            "pdst_name",
            "pdst_id",
            "pdst_is_active_flag",
        ]

        dim_process_stage_definition = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_stage_definition",
            dim_process_stage_definition_final_filters,
            dim_process_stage_definition_selected_fields,
        )

        # ================== dim_process_stage_definition_2 ==================

        dim_process_stage_definition_2_extra_filters = None

        if dim_process_stage_definition_2_extra_filters is not None:
            dim_process_stage_definition_2_extra_filters.append(date_filter)
            dim_process_stage_definition_2_final_filters = " AND ".join(
                dim_process_stage_definition_2_extra_filters
            )
        else:
            dim_process_stage_definition_2_final_filters = date_filter

        dim_process_stage_definition_2_selected_fields = [
            "pdst_name",
            "pdst_id",
            "pdst_is_active_flag",
        ]

        dim_process_stage_definition_2 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_stage_definition",
            dim_process_stage_definition_2_final_filters,
            dim_process_stage_definition_2_selected_fields,
        )

        # ================== dim_process_stage_definition_3 ==================

        dim_process_stage_definition_3_extra_filters = None

        if dim_process_stage_definition_3_extra_filters is not None:
            dim_process_stage_definition_3_extra_filters.append(date_filter)
            dim_process_stage_definition_3_final_filters = " AND ".join(
                dim_process_stage_definition_3_extra_filters
            )
        else:
            dim_process_stage_definition_3_final_filters = date_filter

        dim_process_stage_definition_3_selected_fields = [
            "pdst_name",
            "pdst_id",
            "pdst_is_active_flag",
        ]

        dim_process_stage_definition_3 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_stage_definition",
            dim_process_stage_definition_3_final_filters,
            dim_process_stage_definition_3_selected_fields,
        )

        # ================== dim_process_stage_definition_4 ==================

        dim_process_stage_definition_4_extra_filters = None

        if dim_process_stage_definition_4_extra_filters is not None:
            dim_process_stage_definition_4_extra_filters.append(date_filter)
            dim_process_stage_definition_4_final_filters = " AND ".join(
                dim_process_stage_definition_4_extra_filters
            )
        else:
            dim_process_stage_definition_4_final_filters = date_filter

        dim_process_stage_definition_4_selected_fields = [
            "pdst_name",
            "pdst_id",
            "pdst_is_active_flag",
        ]

        dim_process_stage_definition_4 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_process_stage_definition",
            dim_process_stage_definition_4_final_filters,
            dim_process_stage_definition_4_selected_fields,
        )

        # ================== dim_packet_hier3 ==================

        dim_packet_hier3_extra_filters = None

        if dim_packet_hier3_extra_filters is not None:
            dim_packet_hier3_extra_filters.append(date_filter)
            dim_packet_hier3_final_filters = " AND ".join(
                dim_packet_hier3_extra_filters
            )
        else:
            dim_packet_hier3_final_filters = date_filter

        dim_packet_hier3_selected_fields = [
            "packh3_id",
        ]

        dim_packet_hier3 = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_packet_hier3",
            dim_packet_hier3_final_filters,
            dim_packet_hier3_selected_fields,
        )

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "cust_id",
            "cust_cif",
        ]

        dim_customer_static = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )

        # ================== qca_fsad_case_coc_appeals ==================

        coc_appeals_extra_filters = None

        if coc_appeals_extra_filters is not None:
            coc_appeals_extra_filters.append(date_filter)
            coc_appeals_extra_filters_final_filters = " AND ".join(
                coc_appeals_extra_filters
            )
        else:
            coc_appeals_extra_filters_final_filters = date_filter

        coc_appeals_selected_fields = [
            "CASE_CODE",
            "CCAP_ACTIVE_FLAG",
            "CCAP_CODE",
            "CCAP_COMMITTEE_DECISION_DATE",
            "CCAP_COMMITTEE_DECISION_CODE",
        ]

        coc_appeals = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_fsad_case_coc_appeals",
            coc_appeals_extra_filters_final_filters,
            coc_appeals_selected_fields,
        )

        # ================== qca_at_case_core ================== ## new
 
        at_case_core_extra_filters = None
 
        if at_case_core_extra_filters is not None:
            at_case_core_extra_filters.append(date_filter)
            at_case_core_extra_filters_final_filters = " AND ".join(
                at_case_core_extra_filters
            )
        else:
            at_case_core_extra_filters_final_filters = date_filter
 
        at_case_core_selected_fields = [
            "CASE_CODE",
            "CACO_ADMINISTRATION_STATUS_DATE",
            "CACO_ADMINISTRATION_STATUS_CODE",
        ]
 
        at_case_core = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_at_case_core",
            at_case_core_extra_filters_final_filters,
            at_case_core_selected_fields,
        )
 
        # ================== qca_at_lst_of_val ================== ## new
 
        at_lst_of_val_extra_filters = None
 
        if at_lst_of_val_extra_filters is not None:
            at_lst_of_val_extra_filters.append(date_filter)
            at_lst_of_val_extra_filters_final_filters = " AND ".join(
                at_lst_of_val_extra_filters
            )
        else:
            at_lst_of_val_extra_filters_final_filters = date_filter
 
        at_lst_of_val_selected_fields = [
            "LOV_CODE",
            "LOV_DESC",
        ]
 
        at_lst_of_val = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_at_lst_of_val",
            at_lst_of_val_extra_filters_final_filters,
            at_lst_of_val_selected_fields,
        )
 
        # ================== qca_at_case_extra ================== ## new
 
        at_case_extra_extra_filters = None
 
        if at_case_extra_extra_filters is not None:
            at_case_extra_extra_filters.append(date_filter)
            at_case_extra_extra_filters_final_filters = " AND ".join(
                at_case_extra_extra_filters
            )
        else:
            at_case_extra_extra_filters_final_filters = date_filter
 
        at_case_extra_selected_fields = [
            "CASE_CODE",
            "CAEX_CUSTOM_INT_19",
        ]
 
        at_case_extra = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_at_case_extra",
            at_case_extra_extra_filters_final_filters,
            at_case_extra_selected_fields,
        )
 
        # ================== qca_at_lst_of_val (transaction perimeter) ================== ## new
 
        at_lst_of_val_tp_extra_filters = ["LOVT_CODE = 962"]
 
        if at_lst_of_val_tp_extra_filters is not None:
            at_lst_of_val_tp_extra_filters.append(date_filter)
            at_lst_of_val_tp_extra_filters_final_filters = " AND ".join(
                at_lst_of_val_tp_extra_filters
            )
        else:
            at_lst_of_val_tp_extra_filters_final_filters = date_filter
 
        at_lst_of_val_tp_selected_fields = [
            "LOV_CODE",
            "LOV_DESC",
        ]
 
        at_lst_of_val_tp = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore
            "qca_at_lst_of_val",
            at_lst_of_val_tp_extra_filters_final_filters,
            at_lst_of_val_tp_selected_fields,
        )
 
        # ================== casecodemapping ================== ## new
        case_code_mapping_filter = "1 = 1"
 
 
        casecodemapping_selected_fields = [
            "CASE_CODE_NEW",
            "CASE_CODE_SOR",
            "ACC_NUMBER_NEW",
        ]
 
        casecodemapping = load_landed_delta_table(  # noqa: F841
            spark,
            "reference",  # type: ignore
            "case_codes_mapping",
            case_code_mapping_filter,
            casecodemapping_selected_fields,
        )


        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "PortfolioName",
            "portfolio_id_qca",
       ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== 1st Joins ==================
        # 1st Join conditions
        fs_cases_snap_dim_strategy_step_join_condition = (
            fs_cases_snap["wfst_code_strategy_step"]
            == dim_strategy_step["str_step_id"]  # left
        )

        fs_cases_snap_dim_cases_static_join_condition = (
            fs_cases_snap["case_code"] == dim_cases_static["case_code"]  # inner
        )

        dim_cases_static_fsad_cases_extra_open_join_condition = (
            dim_cases_static["case_code"] == fsad_cases_extra_open["case_code"]  # left
        )

        fs_cases_snap_dim_queue_condition = (
            fs_cases_snap["que_code"] == dim_queue["que_id"]  # left
        )

        fs_cases_snap_fs_cases_snap_extra_condition = (
            fs_cases_snap["case_code"] == fs_cases_snap_extra["case_code"]  # inner
        )

        fs_cases_snap_dim_external_company_condition = (
            fs_cases_snap["extc_code"] == dim_external_company["extc_id"]  # left
        )

        fs_cases_snap_dim_packet_condition = (
            fs_cases_snap["pack_code"] == dim_packet["pack_id"]  # left
        )

        dim_packet_dim_packet_hier3_join_condition = (
            dim_packet["packh3_id"] == dim_packet_hier3["packh3_id"]  # left
        )

        fs_cases_snap_dim_legal_process_status_join_condition = (
            fs_cases_snap["case_legal_process_status"]
            == dim_legal_process_status["lepr_id"]  # left
        )

        # We create the appeal table in order to create our next join
        appeal = fs_cases_snap.filter(
            col("cass_generic_int_03").isin(16449, 16450)
        ).select("case_code")

        appeal = appeal.withColumnRenamed("case_code", "case_code_appeal")

        fs_cases_snap_appeal_join_condition = (
            fs_cases_snap["case_code"] == appeal["case_code_appeal"]  # left
        )

        fs_cases_snap_extra_dim_process_stage_definition_join_condition = (
            fs_cases_snap_extra["cext_generic_int_10"]
            == dim_process_stage_definition["pdst_id"]
        ) & (  # left
            dim_process_stage_definition["pdst_is_active_flag"] == 1
        )

        fs_cases_snap_extra_dim_external_company_2_join_condition = (
            fs_cases_snap_extra["cext_generic_int_09"]
            == dim_external_company_2["extc_id"]  # left
        )

        fs_cases_snap_extra_dim_process_stage_definition_2_join_condition = (
            fs_cases_snap_extra["cext_generic_int_12"]
            == dim_process_stage_definition_2["pdst_id"]
        ) & (  # left
            dim_process_stage_definition_2["pdst_is_active_flag"] == 1
        )

        fs_cases_snap_extra_dim_external_company_3_join_condition = (
            fs_cases_snap_extra["cext_generic_int_11"]
            == dim_external_company_3["extc_id"]  # left
        )

        fs_cases_snap_extra_dim_process_stage_definition_3_join_condition = (
            fs_cases_snap_extra["cext_generic_int_16"]
            == dim_process_stage_definition_3["pdst_id"]
        ) & (  # left
            dim_process_stage_definition_3["pdst_is_active_flag"] == 1
        )

        fs_cases_snap_extra_dim_external_company_4_join_condition = (
            fs_cases_snap_extra["cext_generic_int_15"]
            == dim_external_company_4["extc_id"]  # left
        )

        fs_cases_snap_extra_dim_process_stage_definition_4_join_condition = (
            fs_cases_snap_extra["cext_generic_int_19"]
            == dim_process_stage_definition_4["pdst_id"]
        ) & (  # left
            dim_process_stage_definition_4["pdst_is_active_flag"] == 1
        )

        fs_cases_snap_extra_dim_external_company_5_join_condition = (
            fs_cases_snap_extra["cext_generic_int_18"]
            == dim_external_company_5["extc_id"]  # left
        )

        fs_cases_snap_dim_customer_static_join_condition = (
            fs_cases_snap["cust_code"] == dim_customer_static["cust_id"]  # inner
        )

        fs_cases_snap_portfolio_join_condition = (
            fs_cases_snap["SPET_CODE"] == portfolio["portfolio_id_qca"]
        )

        fs_cases_snap_coc_appeals_join_condition = (
            fs_cases_snap["CASE_CODE"] == coc_appeals["CASE_CODE"]  
        )
        at_case_core_join_condition= (
            fs_cases_snap["case_code"]
            == at_case_core["CASE_CODE"]  # left ## new
        )
        at_lst_of_val_join_condition= (
            at_case_core["CACO_ADMINISTRATION_STATUS_CODE"]
            == at_lst_of_val["LOV_CODE"]  # left ## new
        )
        at_case_extra_join_condition= (
            fs_cases_snap["case_code"]
            == at_case_extra["CASE_CODE"]  # left ## new
        )
        at_lst_of_val_tp_join_condition= (
            at_case_extra["CAEX_CUSTOM_INT_19"]
            == at_lst_of_val_tp["LOV_CODE"]  # left ## new
        )
        casecodemapping_join_condition = (
            fs_cases_snap["CASE_CODE"] == casecodemapping["CASE_CODE_NEW"]  ## new
        )
 

        # Joins
        target_table_qca = (
            fs_cases_snap.join(
                dim_strategy_step,
                fs_cases_snap_dim_strategy_step_join_condition,
                "left",
            )
            .join(
                dim_cases_static,
                fs_cases_snap_dim_cases_static_join_condition,
                "inner",
            )
            .join(
                fsad_cases_extra_open,
                dim_cases_static_fsad_cases_extra_open_join_condition,
                "left",
            )
            .join(dim_queue, fs_cases_snap_dim_queue_condition, "left")
            .join(
                fs_cases_snap_extra,
                fs_cases_snap_fs_cases_snap_extra_condition,
                "inner",
            )
            .join(
                dim_external_company,
                fs_cases_snap_dim_external_company_condition,
                "left",
            )
            .join(dim_packet, fs_cases_snap_dim_packet_condition, "left")
            .join(dim_packet_hier3, dim_packet_dim_packet_hier3_join_condition, "left")
            .join(
                dim_legal_process_status,
                fs_cases_snap_dim_legal_process_status_join_condition,
                "left",
            )
            .join(appeal, fs_cases_snap_appeal_join_condition, "left")
            .join(
                dim_process_stage_definition,
                fs_cases_snap_extra_dim_process_stage_definition_join_condition,
                "left",
            )
            .join(
                dim_external_company_2,
                fs_cases_snap_extra_dim_external_company_2_join_condition,
                "left",
            )
            .join(
                dim_process_stage_definition_2,
                fs_cases_snap_extra_dim_process_stage_definition_2_join_condition,
                "left",
            )
            .join(
                dim_external_company_3,
                fs_cases_snap_extra_dim_external_company_3_join_condition,
                "left",
            )
            .join(
                dim_process_stage_definition_3,
                fs_cases_snap_extra_dim_process_stage_definition_3_join_condition,
                "left",
            )
            .join(
                dim_external_company_4,
                fs_cases_snap_extra_dim_external_company_4_join_condition,
                "left",
            )
            .join(
                dim_process_stage_definition_4,
                fs_cases_snap_extra_dim_process_stage_definition_4_join_condition,
                "left",
            )
            .join(
                dim_external_company_5,
                fs_cases_snap_extra_dim_external_company_5_join_condition,
                "left",
            )
            .join(
                dim_customer_static,
                fs_cases_snap_dim_customer_static_join_condition,
                "inner",
            )
            .join(
                portfolio,
                fs_cases_snap_portfolio_join_condition,
                "left",
            ).join(
                coc_appeals,
                fs_cases_snap_coc_appeals_join_condition,
                "left",
            ).join( ##new
                at_case_core,
                at_case_core_join_condition,
                "left",
            ).join( ##new
                at_lst_of_val,
                at_lst_of_val_join_condition,
                "left",
            ).join( ##new
                at_case_extra,
                at_case_extra_join_condition,
                "left",
            ).join( ##new
                at_lst_of_val_tp,
                at_lst_of_val_tp_join_condition,
                "left",
            ).join( ##new
                casecodemapping,
                casecodemapping_join_condition,
                "left",
            )
        )

        # HashKey
        target_table_qca_CaseCodeId_primary_key_fields = [
            fs_cases_snap.case_code,
            lit("qca"),
        ]

        # Target table qca (hash keys and calculations)
        target_table_qca = (
            target_table_qca.withColumn(
                "CaseCodeId",
                sha2(
                    concat_ws(";", *target_table_qca_CaseCodeId_primary_key_fields),
                    256,
                ),
            )
            .withColumn(
                "WatchlistFlag",
                when(
                    fsad_cases_extra_open["caex_custom_text_29"] == "", lit(None)
                ).otherwise(fsad_cases_extra_open["caex_custom_text_29"]),
            )
            .withColumn(
                "AppealFlag",
                when(appeal["case_code_appeal"].isNotNull(), 1).otherwise(0),
            ).withColumn(
                "CommitteeDecisionTypeCode",
                when(
                    coc_appeals["CCAP_COMMITTEE_DECISION_CODE"].isNull(), 
                    lit(None)
                ).otherwise(
                    when(
                        trim(coc_appeals["CCAP_COMMITTEE_DECISION_CODE"]) == "", 
                        lit(None)
                    ).otherwise(
                        trim(coc_appeals["CCAP_COMMITTEE_DECISION_CODE"]).cast("string")
                    )
                )
            ).withColumn(
                "CommitteeDecisionType",
                when(coc_appeals["CCAP_COMMITTEE_DECISION_CODE"] == 14062, lit("N/A"))
                .when(coc_appeals["CCAP_COMMITTEE_DECISION_CODE"] == 14063, lit("Θετική"))
                .when(coc_appeals["CCAP_COMMITTEE_DECISION_CODE"] == 14064, lit("Απορριπτική"))
                .otherwise(lit(None)) 
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_qca = target_table_qca.select(
            target_table_qca.CaseCodeId.alias("CaseCodeId"),
            lit("qca").cast("string").alias("SourceSystem"),
            portfolio.portfolio_id_qca.cast("string").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("string").alias("BatchId"),
            fs_cases_snap.case_code.cast("int").alias("CaseCodeKey"),
            dim_cases_static.case_code_primary.cast("int").alias("CaseCodePrimary"),
            fs_cases_snap.case_relation_type_code.cast("int").alias(
                "CaseRelationTypeCode"
            ),
            fs_cases_snap.cust_code.cast("int").alias("CustCode"),
            fs_cases_snap.debt_code.cast("int").alias("DebtCode"),
            fs_cases_snap.que_code.cast("int").alias("QueueCode"),
            dim_queue.que_desc.cast("string").alias("Queue"),
            at_case_core.CACO_ADMINISTRATION_STATUS_CODE.cast("int").alias("AdminStatusCode"), #new
            at_lst_of_val["LOV_DESC"].cast("string").alias("AdminStatus"), #new - from at_lst_of_val join
            at_case_core.CACO_ADMINISTRATION_STATUS_DATE.cast("date").alias("AdminStatusDate"), #new
            lit(None).cast("date").alias("AdminStatusExpirationDate"),
            fs_cases_snap.wfst_code_strategy_step.cast("int").alias("StrategyStepCode"),
            dim_strategy_step.str_step_desc.cast("string").alias("StrategyStep"),
            fs_cases_snap.case_source.cast("string").alias("CaseSourceCode"),
            dim_cases_static.case_currently_open.cast("int").alias("CurrentlyOpenFlag"),
            dim_strategy_step.str_steph1_l3_id.cast("int").alias(
                "ParentStrategyStepCode"
            ),
            dim_strategy_step.str_steph1_l3_desc.cast("string").alias(
                "ParentStrategyStep"
            ),
            dim_strategy_step.str_steph3_desc.cast("string").alias("StrategyTypeCode"),
            dim_strategy_step.str_steph3_desc.cast("string").alias("StrategyType"),
            dim_strategy_step.str_steph1_desc.cast("string").alias("StrategyFlowCode"),
            dim_strategy_step.str_steph1_desc.cast("string").alias("StrategyFlow"),
            target_table_qca.WatchlistFlag.cast("string").alias("WatchlistFlag"),
            dim_cases_static.cstm_datetime_ad_03.cast("timestamp").alias(
                "SubsidyEndDate"
            ),
            fs_cases_snap_extra.cext_generic_nvarchar_06.cast("string").alias(
                "LegalFileLocation"
            ),
            fs_cases_snap.caex_custom_datetime_28.cast("timestamp").alias(
                "ApplicationDecisionEverDate"
            ),
            fs_cases_snap_extra.cext_generic_nvarchar_24.cast("string").alias(
                "GovernmentSubsidyType"
            ),
            fs_cases_snap.extc_code.cast("int").alias("AgencyCode"),
            dim_external_company.extc_name.cast("string").alias("Agency"),
            fs_cases_snap.pack_code.cast("int").alias("PacketCode"),
            dim_packet.pack_name.cast("string").alias("Packet"),
            target_table_qca.AppealFlag.cast("int").alias("AppealFlag"),
            fs_cases_snap_extra.cext_generic_int_15.cast("int").alias(
                "ConfiscationCurrentAgencyCode"
            ),
            dim_external_company_4.extc_name.cast("string").alias(
                "ConfiscationCurrentAgency"
            ),
            fs_cases_snap_extra.cext_generic_int_16.cast("int").alias(
                "CurrentStageConfiscationCode"
            ),
            dim_process_stage_definition_3.pdst_name.cast("string").alias(
                "CurrentStageConfiscation"
            ),
            fs_cases_snap_extra.cext_generic_int_11.cast("int").alias(
                "ForcedPrenotationCurrentAgencyCode"
            ),
            dim_external_company_3.extc_name.cast("string").alias(
                "ForcedPrenotationCurrentAgency"
            ),
            fs_cases_snap_extra.cext_generic_int_12.cast("int").alias(
                "CurrentStageForcedPrenotationCode"
            ),
            dim_process_stage_definition_2.pdst_name.cast("string").alias(
                "CurrentStageForcedPrenotation"
            ),
            fs_cases_snap_extra.cext_generic_int_18.cast("int").alias(
                "RegularSuitCurrentAgencyCode"
            ),
            dim_external_company_5.extc_name.cast("string").alias(
                "RegularSuitCurrentAgency"
            ),
            fs_cases_snap_extra.cext_generic_int_19.cast("int").alias(
                "CurrentStageRegularSuitCode"
            ),
            dim_process_stage_definition_4.pdst_name.cast("string").alias(
                "CurrentStageRegularSuit"
            ),
            fs_cases_snap_extra.cext_generic_int_09.cast("int").alias(
                "PaymentOrderCurrentAgencyCode"
            ),
            dim_external_company_2.extc_name.cast("string").alias(
                "PaymentOrderCurrentAgency"
            ),
            fs_cases_snap_extra.cext_generic_int_10.cast("int").alias(
                "CurrentStagePaymentOrderCode"
            ),
            dim_process_stage_definition.pdst_name.cast("string").alias(
                "CurrentStagePaymentOrder"
            ),
            at_lst_of_val_tp["LOV_DESC"].cast("string").alias("TransactionPerimeterCode"), #new - from at_lst_of_val_tp join
            dim_cases_static.CASE_CONTR_NUM.cast("string").alias("AccountNumber"),
            fs_cases_snap_extra.CEXT_DEDICATED_PROPERTY_01_DECIMAL.cast("decimal(19,4)").alias("MinimumPaymentAmount"),
            coc_appeals.CCAP_CODE.cast("int").alias("AppealCode"),
            coc_appeals.CCAP_ACTIVE_FLAG.cast("boolean").alias("ActiveAppealFlag"),
            coc_appeals.CCAP_COMMITTEE_DECISION_DATE.cast("date").alias("DecisionDate"),
            target_table_qca.CommitteeDecisionTypeCode.cast("string").alias("CommitteeDecisionTypeCode"),
            target_table_qca.CommitteeDecisionType.cast("string").alias("CommitteeDecisionType"),
            casecodemapping.CASE_CODE_SOR.cast("int").alias("CaseCodeKeyOld"), ## new
            fs_cases_snap.businessdate.cast("date").alias("BusinessDate"),
        ).distinct() #new

        # ================== Append ==================

        result_df_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_case_code_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)